# YAZEL RECOVAR Integration Demo

This notebook demonstrates how RECOVAR can be used to filter false positive picks from PhaseNet.

- **RECOVAR**: Classifies waveforms to distinguish real earthquakes from noise using learned representations
- **RECOVAR YAZEL Integration**: Uses sliding windows to score PhaseNet picks and filter false positives

#### PhaseNet Configuration:
- **Overlap**: 0.90 (90% overlap between windows)
- **Stacking**: avg (average predictions across overlapping windows)
- **Model**: PhaseNet `instance` pretrained model


In [ ]:
import obspy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time
from datetime import datetime

from yazel_integration_sliding import (
    recovar_pick_cleaner_sliding,
    load_recovar_classifier
)
from demo_plotting import get_phasenet_probabilities, plot_side_by_side_comparison, plot_example, plot_threshold_tradeoff
from demo_utils import load_preprocessed_data, print_confusion_matrix

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

### Setup: Load Model and Data

In [ ]:
# Configuration
PREPROCESSED_DATA_DIR = 'preprocessed_demo_data'
RECOVAR_THRESHOLD = 0.07

# Load preprocessed data (fast - no RECOVAR processing needed!)
print("Loading preprocessed data...")
tp_examples, fp_examples, tp_metadata, fp_metadata = load_preprocessed_data(PREPROCESSED_DATA_DIR)

print(f"Loaded {len(tp_examples)} TRUE PICK examples")
print(f"Loaded {len(fp_examples)} FALSE PICK examples")

# Note: RECOVAR model loading is not needed - scores are precomputed!

## Select Example Picks

For this demonstration, we'll select:
1. **TRUE PICKS**: PhaseNet picks that match catalog events (real earthquakes)
2. **FALSE PICKS**: PhaseNet picks without catalog events (noise/artifacts)

# Data is loaded from preprocessed directories above

**Note:** PhaseNet probability arrays are saved directly in the MSEED files alongside waveform data during the picking phase.

## Visualization

### TRUE PICK Examples (Real Earthquake)

## RECOVAR Filtering Examples

Demonstrating RECOVAR's filtering performance with threshold = 0.07 (max score):
- **TRUE PICK + Kept**: Real earthquake that RECOVAR correctly kept
- **TRUE PICK + Filtered**: Real earthquake that RECOVAR incorrectly rejected  
- **FALSE PICK + Filtered**: Noise/artifact that RECOVAR correctly rejected
- **FALSE PICK + Kept**: Noise/artifact that RECOVAR incorrectly kept

In [ ]:
# Extract RECOVAR scores from metadata (already computed!)
tp_max_scores = tp_metadata['max_score'].tolist()
tp_mean_scores = tp_metadata['mean_score'].tolist()
fp_max_scores = fp_metadata['max_score'].tolist()
fp_mean_scores = fp_metadata['mean_score'].tolist()

print(f"RECOVAR scores loaded for {len(tp_max_scores)} TRUE PICKS and {len(fp_max_scores)} FALSE PICKS")

In [ ]:
# Categorize examples using the threshold
tp_kept = []  # TRUE PICK kept by RECOVAR
tp_filtered = []  # TRUE PICK filtered by RECOVAR
fp_kept = []  # FALSE PICK kept by RECOVAR
fp_filtered = []  # FALSE PICK filtered by RECOVAR

# Categorize TRUE PICKS
for i, example in enumerate(tp_examples):
    max_score = tp_metadata.iloc[i]['max_score']
    example['max_score'] = max_score
    example['phasenet_result'] = get_phasenet_probabilities(example['stream'])
    
    if max_score >= RECOVAR_THRESHOLD:
        tp_kept.append(example)
    else:
        tp_filtered.append(example)

# Categorize FALSE PICKS
for i, example in enumerate(fp_examples):
    max_score = fp_metadata.iloc[i]['max_score']
    example['max_score'] = max_score
    example['phasenet_result'] = get_phasenet_probabilities(example['stream'])
    
    if max_score >= RECOVAR_THRESHOLD:
        fp_kept.append(example)
    else:
        fp_filtered.append(example)

print(f"TRUE PICKS kept by RECOVAR: {len(tp_kept)}")
print(f"TRUE PICKS filtered by RECOVAR: {len(tp_filtered)}")
print(f"FALSE PICKS kept by RECOVAR: {len(fp_kept)}")
print(f"FALSE PICKS filtered by RECOVAR: {len(fp_filtered)}")

# Show multiple examples - different categories side by side
num_examples_to_show = 2

# For visualization, we need to create a mock recovar_result dict
def create_recovar_result(max_score):
    """Create a minimal RECOVAR result dict for plotting"""
    # Create dummy scores array for visualization
    n_windows = 10
    scores = np.linspace(0.01, max_score, n_windows)
    return {
        'max_score': max_score,
        'mean_score': np.mean(scores),
        'scores_array': scores
    }

# Correct decisions: TRUE PICKS KEPT vs FALSE PICKS FILTERED
if tp_kept and fp_filtered:
    print("\n" + "="*80)
    print("CORRECT DECISIONS: TRUE PICKS KEPT (left) vs FALSE PICKS FILTERED (right)")
    print("="*80)
    for i in range(min(num_examples_to_show, len(tp_kept), len(fp_filtered))):
        print(f"\nExample pair {i+1}")
        plot_side_by_side_comparison(
            tp_kept[i], create_recovar_result(tp_kept[i]['max_score']), tp_kept[i]['phasenet_result'],
            fp_filtered[i], create_recovar_result(fp_filtered[i]['max_score']), fp_filtered[i]['phasenet_result'],
            threshold=RECOVAR_THRESHOLD
        )

# Incorrect decisions: TRUE PICKS FILTERED vs FALSE PICKS KEPT
if tp_filtered and fp_kept:
    print("\n" + "="*80)
    print("INCORRECT DECISIONS: TRUE PICKS FILTERED (left) vs FALSE PICKS KEPT (right)")
    print("="*80)
    for i in range(min(num_examples_to_show, len(tp_filtered), len(fp_kept))):
        print(f"\nExample pair {i+1}")
        plot_side_by_side_comparison(
            tp_filtered[i], create_recovar_result(tp_filtered[i]['max_score']), tp_filtered[i]['phasenet_result'],
            fp_kept[i], create_recovar_result(fp_kept[i]['max_score']), fp_kept[i]['phasenet_result'],
            threshold=RECOVAR_THRESHOLD
        )

# If we only have one type of incorrect decision, show it anyway
if tp_filtered and not fp_kept:
    print("\n" + "="*80)
    print("FALSE NEGATIVES: TRUE PICKS FILTERED BY RECOVAR")
    print("="*80)
    for i in range(min(num_examples_to_show, len(tp_filtered))):
        print(f"\nExample {i+1}")
        plot_example(tp_filtered[i], create_recovar_result(tp_filtered[i]['max_score']), 
                    tp_filtered[i]['phasenet_result'], threshold=RECOVAR_THRESHOLD)

if fp_kept and not tp_filtered:
    print("\n" + "="*80)
    print("FALSE POSITIVES: FALSE PICKS KEPT BY RECOVAR")
    print("="*80)
    for i in range(min(num_examples_to_show, len(fp_kept))):
        print(f"\nExample {i+1}")
        plot_example(fp_kept[i], create_recovar_result(fp_kept[i]['max_score']), 
                    fp_kept[i]['phasenet_result'], threshold=RECOVAR_THRESHOLD)

### Performance Metrics at Recommended Threshold (0.07)

In [ ]:
# Display confusion matrix at recommended threshold
print_confusion_matrix(tp_kept, tp_filtered, fp_kept, fp_filtered, RECOVAR_THRESHOLD)

### Threshold Trade-off Analysis

This plot shows how different thresholds affect the filtering performance:
- **Missed True Positives**: Real earthquakes that would be filtered out (False Negatives)
- **Filtered False Positives**: Noise that would be correctly removed (True Negatives)

In [ ]:
# Plot threshold trade-off analysis
plot_threshold_tradeoff(tp_max_scores, fp_max_scores, RECOVAR_THRESHOLD)

## Summary
- A well-chosen RECOVAR threshold can filter many false positives while retaining most true positives

For batch processing and full evaluation, see `run_yazel_batch_sliding.py`